In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Import Essential Libraries for the entire project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score)
import time

# For saving our best model
import joblib

# Magic command for plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6) # Set default plot size
sns.set_style("whitegrid")
np.random.seed(42) # For reproducibility

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive')

# Import necessary libraries
import pandas as pd
import numpy as np
import os

# Define your folder paths
ml_folder = '/content/drive/MyDrive/CICIDS/CSVs/ML/'
flows_folder = '/content/drive/MyDrive/CICIDS/CSVs/Flows/'

# List all files in the ML folder
print("Files in ML folder:")
ml_files = os.listdir(ml_folder)
for file in ml_files:
    print(f"  - {file}")

# List all files in the Flows folder
print("\nFiles in Flows folder:")
flows_files = os.listdir(flows_folder)
for file in flows_files:
    print(f"  - {file}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in ML folder:
  - Wednesday-workingHours.pcap_ISCX.csv
  - Tuesday-WorkingHours.pcap_ISCX.csv
  - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Monday-WorkingHours.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Files in Flows folder:
  - Wednesday-workingHours.pcap_ISCX.csv
  - Tuesday-WorkingHours.pcap_ISCX.csv
  - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Monday-WorkingHours.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv


In [5]:
# Combine all CSV files from the ML folder
all_dataframes = []

print("Reading and combining files...")
for file in ml_files:
    if file.endswith('.csv'):
        file_path = os.path.join(ml_folder, file)
        print(f"Reading {file}...")

        # Read the CSV file
        df = pd.read_csv(file_path)
        all_dataframes.append(df)

        print(f"  - Shape: {df.shape}")
        print(f"  - Columns: {list(df.columns)}")

# Combine all dataframes
print("\nCombining all files...")
combined_df = pd.concat(all_dataframes, ignore_index=True)

print(f"Final combined dataset shape: {combined_df.shape}")
print(f"Total files combined: {len(all_dataframes)}")

# Show first few rows
combined_df.head()

Reading and combining files...
Reading Wednesday-workingHours.pcap_ISCX.csv...
  - Shape: (692703, 79)
  - Columns: [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length 

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [6]:
# Save the combined dataset so we don't have to combine again
combined_file_path = '/content/drive/MyDrive/CICIDS/combined_dataset.csv'
combined_df.to_csv(combined_file_path, index=False)
print(f"Combined dataset saved to: {combined_file_path}")
print(f"File size: {combined_df.shape}")

Combined dataset saved to: /content/drive/MyDrive/CICIDS/combined_dataset.csv
File size: (2830743, 79)


In [7]:
# Let's examine one file from the Flows folder to understand its structure
if flows_files:  # If there are files in the flows folder
    # Take the first CSV file from flows folder
    sample_flow_file = os.path.join(flows_folder, flows_files[0])
    print(f"Examining: {flows_files[0]}")

    # Read just the first few rows to see the structure
    flow_sample = pd.read_csv(sample_flow_file, nrows=5)
    print(f"Shape: {flow_sample.shape}")
    print("\nColumns in flows data:")
    print(flow_sample.columns.tolist())
    print("\nFirst 2 rows:")
    print(flow_sample.head(2))

    # Check if there's a label column in flows data
    flow_label_candidates = [col for col in flow_sample.columns if 'label' in col.lower() or 'attack' in col.lower()]
    print(f"\nLabel columns in flows data: {flow_label_candidates}")
else:
    print("No files found in Flows folder")

Examining: Wednesday-workingHours.pcap_ISCX.csv
Shape: (5, 85)

Columns in flows data:
['Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' P

In [8]:
# Let's thoroughly explore the Flows folder structure
print("=== COMPREHENSIVE FLOWS FOLDER ANALYSIS ===\n")

# Check how many files and what types
all_flow_files = os.listdir(flows_folder)
csv_files = [f for f in all_flow_files if f.endswith('.csv')]
pcap_files = [f for f in all_flow_files if f.endswith('.pcap')]
other_files = [f for f in all_flow_files if f not in csv_files + pcap_files]

print(f"Total files in Flows folder: {len(all_flow_files)}")
print(f"CSV files: {len(csv_files)}")
print(f"PCAP files: {len(pcap_files)}")
print(f"Other files: {len(other_files)}")

# Display the CSV files (these are what we'll work with)
print(f"\nCSV Files in Flows folder:")
for i, file in enumerate(csv_files, 1):
    print(f"  {i}. {file}")

# Let's examine the first CSV file to understand the structure
if csv_files:
    first_csv_path = os.path.join(flows_folder, csv_files[0])
    print(f"\n=== ANALYZING: {csv_files[0]} ===")

    # Load a small sample to understand structure
    sample_data = pd.read_csv(first_csv_path, nrows=10)
    print(f"Shape: {sample_data.shape} (rows, columns)")
    print(f"Columns: {len(sample_data.columns)}")

    print("\nColumn Names:")
    for i, col in enumerate(sample_data.columns, 1):
        print(f"  {i:2d}. {col}")

    print("\nFirst 3 rows:")
    display(sample_data.head(3))

    # Check for label columns
    label_cols = [col for col in sample_data.columns if 'label' in col.lower() or 'class' in col.lower() or 'attack' in col.lower()]
    print(f"\nPotential label columns: {label_cols}")

    if label_cols:
        print(f"Unique values in '{label_cols[0]}': {sample_data[label_cols[0]].unique()}")
else:
    print("No CSV files found in Flows folder!")

=== COMPREHENSIVE FLOWS FOLDER ANALYSIS ===

Total files in Flows folder: 8
CSV files: 8
PCAP files: 0
Other files: 0

CSV Files in Flows folder:
  1. Wednesday-workingHours.pcap_ISCX.csv
  2. Tuesday-WorkingHours.pcap_ISCX.csv
  3. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  4. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  5. Monday-WorkingHours.pcap_ISCX.csv
  6. Friday-WorkingHours-Morning.pcap_ISCX.csv
  7. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  8. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

=== ANALYZING: Wednesday-workingHours.pcap_ISCX.csv ===
Shape: (10, 85) (rows, columns)
Columns: 85

Column Names:
   1. Flow ID
   2.  Source IP
   3.  Source Port
   4.  Destination IP
   5.  Destination Port
   6.  Protocol
   7.  Timestamp
   8.  Flow Duration
   9.  Total Fwd Packets
  10.  Total Backward Packets
  11. Total Length of Fwd Packets
  12.  Total Length of Bwd Packets
  13.  Fwd Packet Length Max
  14.  Fwd Packet Length Min
 

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.14-209.48.71.168-49459-80-6,192.168.10.14,49459,209.48.71.168,80,6,5/7/2017 8:42,38308,1,1,...,20,0,0,0,0,0,0,0,0,BENIGN
1,192.168.10.3-192.168.10.17-389-49453-6,192.168.10.17,49453,192.168.10.3,389,6,5/7/2017 8:42,479,11,5,...,32,0,0,0,0,0,0,0,0,BENIGN
2,192.168.10.3-192.168.10.17-88-46124-6,192.168.10.17,46124,192.168.10.3,88,6,5/7/2017 8:42,1095,10,6,...,32,0,0,0,0,0,0,0,0,BENIGN



Potential label columns: [' Label']
Unique values in ' Label': ['BENIGN']


In [11]:
# First, let's detect the correct encoding for each file
import chardet

print("=== DETECTING FILE ENCODINGS ===\n")

file_encodings = {}

for file in csv_files:
    file_path = os.path.join(flows_folder, file)

    # Detect encoding by reading a sample of the file
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)  # Read first 10KB to detect encoding
        result = chardet.detect(raw_data)

    file_encodings[file] = result['encoding']
    print(f"{file}: {result['encoding']} (confidence: {result['confidence']:.2f})")

# Show summary
print(f"\nMost common encoding: {pd.Series(file_encodings.values()).mode()[0]}")

=== DETECTING FILE ENCODINGS ===

Wednesday-workingHours.pcap_ISCX.csv: ascii (confidence: 1.00)
Tuesday-WorkingHours.pcap_ISCX.csv: ascii (confidence: 1.00)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: ascii (confidence: 1.00)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: ascii (confidence: 1.00)
Monday-WorkingHours.pcap_ISCX.csv: ascii (confidence: 1.00)
Friday-WorkingHours-Morning.pcap_ISCX.csv: ascii (confidence: 1.00)
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: ascii (confidence: 1.00)
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: ascii (confidence: 1.00)

Most common encoding: ascii


In [12]:
# Now read files with proper encoding handling
print("=== READING FILES WITH CORRECT ENCODING ===\n")

all_flow_dfs = []
problem_files = []

for file in csv_files:
    file_path = os.path.join(flows_folder, file)
    print(f"Reading {file}...")

    try:
        # Try reading with detected encoding first
        encoding = file_encodings[file]
        df = pd.read_csv(file_path, encoding=encoding, on_bad_lines='warn')
        all_flow_dfs.append(df)
        print(f"  ✓ Success with {encoding} - Shape: {df.shape}")

    except Exception as e:
        print(f"  ✗ Failed with {encoding}: {e}")

        # Try alternative encodings
        for alt_encoding in ['latin-1', 'iso-8859-1', 'cp1252']:
            try:
                df = pd.read_csv(file_path, encoding=alt_encoding, on_bad_lines='warn')
                all_flow_dfs.append(df)
                print(f"  ✓ Success with {alt_encoding} - Shape: {df.shape}")
                break
            except:
                continue
        else:
            print(f"  ✗ Could not read {file} with any encoding")
            problem_files.append(file)

print(f"\nSuccessfully read {len(all_flow_dfs)} files")
if problem_files:
    print(f"Could not read {len(problem_files)} files: {problem_files}")

=== READING FILES WITH CORRECT ENCODING ===

Reading Wednesday-workingHours.pcap_ISCX.csv...
  ✓ Success with ascii - Shape: (692703, 85)
Reading Tuesday-WorkingHours.pcap_ISCX.csv...
  ✓ Success with ascii - Shape: (445909, 85)
Reading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
  ✗ Failed with ascii: 'ascii' codec can't decode byte 0x96 in position 22398: ordinal not in range(128)


/tmp/ipython-input-1260297640.py:24: DtypeWarning: Columns (0,1,3,6,84) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding=alt_encoding, on_bad_lines='warn')


  ✓ Success with latin-1 - Shape: (458968, 85)
Reading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
  ✓ Success with ascii - Shape: (288602, 85)
Reading Monday-WorkingHours.pcap_ISCX.csv...
  ✓ Success with ascii - Shape: (529918, 85)
Reading Friday-WorkingHours-Morning.pcap_ISCX.csv...
  ✓ Success with ascii - Shape: (191033, 85)
Reading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
  ✓ Success with ascii - Shape: (225745, 85)
Reading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
  ✓ Success with ascii - Shape: (286467, 85)

Successfully read 8 files


In [15]:
# Combine all successfully read dataframes
if all_flow_dfs:
    print("=== COMBINING SUCCESSFULLY READ FILES ===\n")

    # Check if all dataframes have the same columns
    all_columns = [set(df.columns) for df in all_flow_dfs]
    common_columns = set.intersection(*all_columns)

    print(f"Common columns across all files: {len(common_columns)}")
    print(f"Sample columns: {list(common_columns)[:10]}...")

    # Keep only common columns to avoid issues
    standardized_dfs = []
    for df in all_flow_dfs:
        standardized_df = df[[col for col in df.columns if col in common_columns]]
        standardized_dfs.append(standardized_df)

    # Combine all dataframes
    flows_combined = pd.concat(standardized_dfs,  ignore_index=True)

    print(f"Final combined dataset shape: {flows_combined.shape}")
    print(f"Total rows: {len(flows_combined)}")

    # Show basic info
    print("\n=== COMBINED DATASET INFO ===")
    print(f"Columns: {list(flows_combined.columns)}")
    print(f"Label column values: {flows_combined[' Label'].value_counts()}")

    # Display first few rows
    print("\nFirst 3 rows:")
    display(flows_combined.head(3))

else:
    print("No files were successfully read!")

=== COMBINING SUCCESSFULLY READ FILES ===

Common columns across all files: 85
Sample columns: [' Subflow Bwd Packets', ' Init_Win_bytes_backward', ' Fwd Packet Length Min', 'Flow ID', ' CWE Flag Count', ' SYN Flag Count', ' Timestamp', ' Idle Max', ' Fwd URG Flags', ' Down/Up Ratio']...
Final combined dataset shape: (3119345, 85)
Total rows: 3119345

=== COMBINED DATASET INFO ===
Columns: ['Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.14-209.48.71.168-49459-80-6,192.168.10.14,49459.0,209.48.71.168,80.0,6.0,5/7/2017 8:42,38308.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,192.168.10.3-192.168.10.17-389-49453-6,192.168.10.17,49453.0,192.168.10.3,389.0,6.0,5/7/2017 8:42,479.0,11.0,5.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,192.168.10.3-192.168.10.17-88-46124-6,192.168.10.17,46124.0,192.168.10.3,88.0,6.0,5/7/2017 8:42,1095.0,10.0,6.0,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [16]:
# Save our progress so far
if 'flows_combined' in locals() and len(flows_combined) > 0:
    print("=== SAVING COMBINED DATASET ===\n")

    # Create a temporary sample first to test
    sample_path = '/content/drive/MyDrive/CICIDS/flows_sample.csv'
    flows_combined.head(1000).to_csv(sample_path, index=False)
    print(f"Sample saved to: {sample_path}")

    # Now save the full dataset in chunks
    full_path = '/content/drive/MyDrive/CICIDS/combined_flows_dataset.csv'
    chunk_size = 50000  # Smaller chunks to avoid memory issues

    total_rows = len(flows_combined)
    print(f"Saving {total_rows:,} rows in chunks of {chunk_size}...")

    for i, start in enumerate(range(0, total_rows, chunk_size)):
        end = min(start + chunk_size, total_rows)
        chunk = flows_combined.iloc[start:end]

        if i == 0:
            chunk.to_csv(full_path, index=False)
        else:
            chunk.to_csv(full_path, mode='a', header=False, index=False)

        if (i + 1) % 10 == 0:  # Print progress every 10 chunks
            print(f"Saved {end:,} rows...")

    print(f"✓ Full dataset saved to: {full_path}")
    print(f"File size: {os.path.getsize(full_path) / 1024**3:.2f} GB")

else:
    print("No data to save!")

=== SAVING COMBINED DATASET ===

Sample saved to: /content/drive/MyDrive/CICIDS/flows_sample.csv
Saving 3,119,345 rows in chunks of 50000...
Saved 500,000 rows...
Saved 1,000,000 rows...
Saved 1,500,000 rows...
Saved 2,000,000 rows...
Saved 2,500,000 rows...
Saved 3,000,000 rows...
✓ Full dataset saved to: /content/drive/MyDrive/CICIDS/combined_flows_dataset.csv
File size: 1.46 GB


In [3]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("All libraries imported successfully!")

All libraries imported successfully!


Loading ML dataset from: /content/drive/MyDrive/CICIDS/combined_dataset.csv
✓ ML Dataset loaded successfully!
ML Dataset shape: (2830743, 79) (rows, columns)

=== ML DATASET EXPLORATION ===
First 3 rows:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN



Column names (first 20):
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min']

✓ Label column found: ' Label'
Unique values: ['BENIGN' 'DoS slowloris' 'DoS Slowhttptest' 'DoS Hulk' 'DoS GoldenEye'
 'Heartbleed' 'FTP-Patator' 'SSH-Patator' 'Web Attack � Brute Force'
 'Web Attack � XSS' 'Web Attack � Sql Injection' 'Infiltration' 'PortScan'
 'Bot' 'DDoS']
Value counts:
 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator         

In [1]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("All libraries imported successfully!")

# Check if we have flows data available
flows_combined_path = '/content/drive/MyDrive/CICIDS/combined_flows_dataset.csv'
flows_sample_path = '/content/drive/MyDrive/CICIDS/flows_sample.csv'

print("Checking for flows data...")

if os.path.exists(flows_combined_path):
    print("✓ Found combined flows dataset!")
    flows_combined = pd.read_csv(flows_combined_path)
    print(f"Flows dataset shape: {flows_combined.shape}")
    print(f"Label values: {flows_combined[' Label'].value_counts()}")

elif os.path.exists(flows_sample_path):
    print("✓ Found flows sample dataset!")
    flows_combined = pd.read_csv(flows_sample_path)
    print(f"Flows sample shape: {flows_combined.shape}")
    print(f"Label values: {flows_combined[' Label'].value_counts()}")

else:
    print(" No flows dataset found yet")
    print("We'll work with ML data first")

All libraries imported successfully!
Checking for flows data...
✓ Found combined flows dataset!


/tmp/ipython-input-3911852338.py:24: DtypeWarning: Columns (0,1,3,6,84) have mixed types. Specify dtype option on import or set low_memory=False.
  flows_combined = pd.read_csv(flows_combined_path)


Flows dataset shape: (3119345, 85)
Label values:  Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack  Brute Force         1507
Web Attack  XSS                  652
Infiltration                       36
Web Attack  Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64
